
# UD4 — Apéndice  
## Primera red neuronal con **JAX** (Fashion-MNIST)

Este notebook muestra **la misma idea** que en Keras y PyTorch,
pero usando **JAX**.

### Objetivo
- Entender el enfoque **funcional**
- Ver cómo se calculan gradientes con `jax.grad`
- Comprender por qué JAX se usa en investigación y modelos grandes

⚠️ Este notebook es **introductorio**, no exhaustivo.


# Introducción a JAX

JAX se basa en programación funcional y derivación automática.
Sus ideas clave son:

1. Funciones puras
2. Gradientes con `jax.grad`
3. Aceleración con `jax.jit`

En este apéndice aplicamos la misma idea de red neuronal para comparar con Keras y PyTorch.


## Problema a resolver

Usamos **Fashion-MNIST** para resolver una tarea de clasificación con una red densa sencilla.

- Entrada: imagen 28x28
- Salida: clase predicha
- Entrenamiento: actualización de parámetros con gradientes calculados por JAX

El foco es entender la mecánica funcional del entrenamiento.


## Mini-guia JAX (resumen docente)

### Qué es JAX
JAX combina cálculo numérico con derivación automática y compilación (`jit`).

Ideas clave:
- Programación funcional
- Gradientes con `jax.grad`
- Vectorización con `vmap`
- Compilación con `jit`

### Flujo típico
1. Definir función de forward
2. Definir función de pérdida
3. Obtener gradientes con `grad`
4. Actualizar parámetros de forma funcional

### En qué destaca
- Investigación y rendimiento
- Código matemático explícito
- Escalado en aceleradores (según stack)

### Precauciones docentes
- Curva de aprendizaje mayor que Keras/PyTorch
- Evitar complejidad de más en primera toma

### Regla práctica
Mismo problema que en Keras/PyTorch para comparar, cambiando solo el estilo
(funcional) y el mecanismo de gradiente.



## Entorno de ejecución (CPU/GPU)

Este notebook está preparado para ejecutarse en:

- CPU local
- GPU local (si está configurada)
- Google Colab (recomendado si no hay GPU local)

Recomendación docente:
- Primera ejecución en CPU para entender el flujo.
- Segunda ejecución en GPU para comparar tiempos.


In [ ]:
import jax

print("JAX:", jax.__version__)
print("Dispositivos disponibles:", jax.devices())



## 1. Carga y preparación de datos


In [ ]:

(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

shoe_classes = [5, 7, 9]

y_train_bin = np.isin(y_train, shoe_classes).astype(np.float32)
y_test_bin = np.isin(y_test, shoe_classes).astype(np.float32)

x_train = x_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0

x_train = x_train.reshape(-1, 28*28)
x_test = x_test.reshape(-1, 28*28)

X_train = jnp.array(x_train)
y_train_j = jnp.array(y_train_bin)



## 2. Inicialización de parámetros

En JAX:
- No hay clases de modelo
- Trabajamos con **parámetros explícitos**


In [ ]:

key = jax.random.PRNGKey(0)

def init_params(key):
    k1, k2 = jax.random.split(key)
    W1 = jax.random.normal(k1, (784, 128)) * 0.01
    b1 = jnp.zeros((128,))
    W2 = jax.random.normal(k2, (128, 1)) * 0.01
    b2 = jnp.zeros((1,))
    return (W1, b1, W2, b2)

params = init_params(key)



## 3. Definición del modelo (forward)


In [ ]:

def relu(x):
    return jnp.maximum(0, x)

def sigmoid(x):
    return 1 / (1 + jnp.exp(-x))

def forward(params, x):
    W1, b1, W2, b2 = params
    h = relu(jnp.dot(x, W1) + b1)
    out = sigmoid(jnp.dot(h, W2) + b2)
    return out.squeeze()



## 4. Función de pérdida


In [ ]:

def binary_cross_entropy(y_true, y_pred):
    eps = 1e-7
    return -jnp.mean(y_true * jnp.log(y_pred + eps) +
                     (1 - y_true) * jnp.log(1 - y_pred + eps))

def loss_fn(params, x, y):
    preds = forward(params, x)
    return binary_cross_entropy(y, preds)



## 5. Gradientes y actualización de parámetros


In [ ]:

grad_fn = jax.grad(loss_fn)

def update(params, grads, lr=0.01):
    return tuple(p - lr * g for p, g in zip(params, grads))



## 6. Entrenamiento (muy simplificado)


In [ ]:

epochs = 10
for epoch in range(epochs):
    grads = grad_fn(params, X_train, y_train_j)
    params = update(params, grads)
    loss = loss_fn(params, X_train, y_train_j)
    print(f"Epoch {epoch+1} - Loss: {loss:.4f}")



## 7. Conclusiones sobre JAX

- Todo es explícito
- El modelo es una función
- Los gradientes se calculan automáticamente
- Ideal para investigación y alto rendimiento

En producción general y aprendizaje inicial,
Keras y PyTorch suelen ser más adecuados.
